In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("medical_dataset.csv")    

In [5]:
df.head()

,age,weight,blood_pressure,glucose_level,insulin_level,smoking_habit,symptom_1,symptom_2,symptom_3,disease
0,68,72,140/90,190,30,yes,memory_loss,confusion,disorientation,Alzheimer’s
1,54,80,135/85,105,20,no,tremors,stiff_muscles,speech_issue,Parkinson’s
2,36,65,125/80,180,22,yes,blurred_vision,fatigue,frequent_urination,Diabetes
3,29,60,118/76,95,10,no,joint_pain,rash,fever,Lupus
4,41,70,130/82,100,18,yes,headache,dizziness,nausea,Migraine


In [6]:
list1 = ["smoking_habit"]
df[list1]=df[list1].replace({"no":0,"yes":1})

In [7]:
df.head()

,age,weight,blood_pressure,glucose_level,insulin_level,smoking_habit,symptom_1,symptom_2,symptom_3,disease
0,68,72,140/90,190,30,1,memory_loss,confusion,disorientation,Alzheimer’s
1,54,80,135/85,105,20,0,tremors,stiff_muscles,speech_issue,Parkinson’s
2,36,65,125/80,180,22,1,blurred_vision,fatigue,frequent_urination,Diabetes
3,29,60,118/76,95,10,0,joint_pain,rash,fever,Lupus
4,41,70,130/82,100,18,1,headache,dizziness,nausea,Migraine


Rows, columns

In [8]:
df.shape

(82, 10)

Missing values

In [9]:
df.isnull().sum()

age               0
weight            0
blood_pressure    0
glucose_level     0
insulin_level     0
smoking_habit     0
symptom_1         0
symptom_2         0
symptom_3         0
disease           0
dtype: int64

Disease counts

In [10]:
df["disease"].value_counts()

disease
Parkinson’s               11
Diabetes                  10
Lupus                     10
Cardiovascular Disease    10
Alzheimer’s                9
Migraine                   9
Stroke                     9
General Anxiety            7
Meningitis                 6
Parkinson's                1
Name: count, dtype: int64

Change disease name one(there Parkinson's  is defined in 2 times. see apostrophe)

In [11]:
df["disease"] = (
    df["disease"]
    .str.replace("\u2019", "'", regex=False)    #\u2019 means right single quotation mark("’") this is change to '
    .str.strip()
)

In [12]:
df["disease"].value_counts()

disease
Parkinson's               12
Diabetes                  10
Lupus                     10
Cardiovascular Disease    10
Alzheimer's                9
Migraine                   9
Stroke                     9
General Anxiety            7
Meningitis                 6
Name: count, dtype: int64

put input and output

In [13]:
symptom_cols = ["symptom_1", "symptom_2", "symptom_3"]
X = (
    df[symptom_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace("_", " ")
    .str.lower()
)
y = df["disease"]

In [14]:
X

0          memory loss confusion disorientation
1            tremors stiff muscles speech issue
2     blurred vision fatigue frequent urination
3                         joint pain rash fever
4                     headache dizziness nausea
                        ...                    
77            chest tightness dizziness fatigue
78             tremors slow movement depression
79                     anxiety insomnia fatigue
80                           fever rash fatigue
81         speech difficulty numbness confusion
Length: 82, dtype: str

In [15]:
y

0                Alzheimer's
1                Parkinson's
2                   Diabetes
3                      Lupus
4                   Migraine
               ...          
77    Cardiovascular Disease
78               Parkinson's
79           General Anxiety
80                     Lupus
81                    Stroke
Name: disease, Length: 82, dtype: str

In [16]:
from sklearn.model_selection import train_test_split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

Imports

In [27]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np
import joblib

Compare models (cross-validation)

In [28]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
}
for name, clf in models.items():
    pipe = Pipeline([("vec", CountVectorizer(binary=True)), ("clf", clf)])
    s = cross_val_score(pipe, X, y, cv=cv)
    print(f"{name}: {s.mean()*100:.2f}% (+/- {s.std()*100:.2f})")

Naive Bayes: 95.00% (+/- 4.68)
Logistic Regression: 96.25% (+/- 3.06)
Random Forest: 95.07% (+/- 2.47)


Final model (Logistic Regression) + test accuracy

In [29]:
model = Pipeline([
    ("vectorizer", CountVectorizer(binary=True)),
    ("classifier", LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print(classification_report(y_test, predictions, zero_division=0))

Test Accuracy: 95.24%
                        precision    recall  f1-score   support

           Alzheimer's       0.67      1.00      0.80         2
Cardiovascular Disease       1.00      1.00      1.00         3
              Diabetes       1.00      1.00      1.00         3
       General Anxiety       1.00      1.00      1.00         2
                 Lupus       1.00      1.00      1.00         3
            Meningitis       1.00      1.00      1.00         1
              Migraine       1.00      1.00      1.00         2
           Parkinson's       1.00      1.00      1.00         3
                Stroke       1.00      0.50      0.67         2

              accuracy                           0.95        21
             macro avg       0.96      0.94      0.94        21
          weighted avg       0.97      0.95      0.95        21



Cross-validation of final model

In [30]:
scores = cross_val_score(model, X, y, cv=cv)
print(scores)
print(f"CV Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f})")

[1.     1.     0.9375 0.9375 0.9375]
CV Accuracy: 96.25% (+/- 3.06)


In [31]:
import joblib
model.fit(X, y)          # final model: full data-la train
joblib.dump(model, "disease_model.pkl")

['disease_model.pkl']

The Interactive input() Box. give a symtoms and predict the disease

In [99]:
# 1. Ask the user for symptoms
user_input = input("Enter your symptoms (e.g. tremors, stiff muscles, speech issue): ")

# 2. Clean the input (lowercase and replace underscores)
cleaned_input = user_input.replace("_", " ").lower()

# 3. Predict disease
predicted_disease = model.predict([cleaned_input])[0]

print("\n------------------------------")
print(f"Symptoms Entered: {user_input}")
print(f"Predicted Diagnosis: {predicted_disease}")
print("------------------------------")


------------------------------
Symptoms Entered: 
Predicted Diagnosis: Parkinson's
------------------------------


Enter 3 Separate Symptoms (Matching Dataset Format)

In [100]:
s1 = input("Enter Symptom 1: ")
s2 = input("Enter Symptom 2: ")
s3 = input("Enter Symptom 3: ")
print(f"Symptoms Entered: {s1,s2,s3}")

# Combine into one text string
combined_symptoms = f"{s1} {s2} {s3}".replace("_", " ").lower()

# Predict
predicted_disease = model.predict([combined_symptoms])[0]

print(f"\nPredicted Disease: {predicted_disease}")

Symptoms Entered: ('', '', '')

Predicted Disease: Parkinson's


Show Diagnosis with Probability / Confidence %

In [101]:
import numpy as np

user_input = input("Enter symptoms: ")
cleaned = user_input.replace("_", " ").lower()

# Get probability for all 9 diseases
probabilities = model.predict_proba([cleaned])[0]
top_idx = np.argmax(probabilities)

predicted_disease = model.classes_[top_idx]
confidence = probabilities[top_idx] * 100

print(f"\nPrimary Diagnosis : {predicted_disease} ({confidence:.1f}% confidence)")

# Show top 3 most likely candidates
print("\nTop 3 Possibilities:")
top_3_indices = np.argsort(probabilities)[::-1][:3]
for rank, idx in enumerate(top_3_indices, 1):
    print(f"  {rank}. {model.classes_[idx]} ({probabilities[idx] * 100:.1f}%)")


Primary Diagnosis : Parkinson's (14.8% confidence)

Top 3 Possibilities:
  1. Parkinson's (14.8%)
  2. Stroke (11.5%)
  3. Migraine (11.5%)
